# Entity Linking and Knowledge Representation

Owner: Member 3

This notebook maps extracted claim triples from Notebook 1 into a knowledge graph-compatible representation for downstream KG reasoning.

## Role in the Project

Notebook 1 produces extracted triples such as `subject`, `relation`, and `object`. This notebook links entity-like subjects and objects to Wikidata identifiers and maps selected relations to Wikidata property IDs when the mapping is defensible.

The output is `03_Entity_Linking_KR/linked_entities.json`, which is the expected input for `04_KG_Reasoning`.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'LIAR_dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODULE_DIR = PROJECT_ROOT / '03_Entity_Linking_KR'
sys.path.insert(0, str(MODULE_DIR))

from run_entity_linking import normalize_input_records, run_entity_linking

INPUT_PATH = PROJECT_ROOT / '01_Claim_Extraction' / 'extracted_triples_filtered.json'
OUTPUT_PATH = MODULE_DIR / 'linked_entities.json'
SUMMARY_PATH = MODULE_DIR / 'entity_linking_summary.json'

INPUT_PATH, OUTPUT_PATH, SUMMARY_PATH

## Input Schema

The preferred input is `01_Claim_Extraction/extracted_triples_filtered.json`. If Notebook 1 does not provide `claim_id`, this module creates stable IDs such as `claim-00001`. If Notebook 1 uses `confidence`, this module normalizes it to `extraction_confidence`.

In [ ]:
records = json.loads(INPUT_PATH.read_text())
df = normalize_input_records(records)
df.head()

## Method

The entity-linking method uses the following safeguards:

- link only entity-like values: people, organisations, locations, geopolitical entities, and miscellaneous named entities
- keep dates, percentages, counts, and money values as literal objects
- expand common ambiguous political names before search, such as `Obama -> Barack Obama`, `McCain -> John McCain`, and `ISIS -> Islamic State`
- reject obvious bad Wikidata hits such as family-name pages, given-name pages, disambiguation pages, and unrelated academic journals
- map `property_id` only when the relation or claim pattern gives a defensible Wikidata property

In [ ]:
# Full run. This calls the Wikidata API for entity search.
# For a fast smoke test without network calls, add: offline=True, max_records=20
results = run_entity_linking(PROJECT_ROOT, sleep_seconds=0.05)
summary = results['summary']
summary

## Linked Output Preview

Each output row preserves the original claim fields and adds Wikidata entity IDs, labels, descriptions, relation property metadata, confidence, and notes.

In [ ]:
linked_df = results['linked_df']
linked_df.head()

## Handoff Validation

The fields below are required by the shared project schema and should be present for every record before moving to KG reasoning.

In [ ]:
required_fields = [
    'claim_id',
    'raw_claim',
    'subject',
    'subject_kg_id',
    'object',
    'object_kg_id',
    'relation',
    'property_id',
    'linking_confidence',
    'linking_notes',
]

missing = {field: int(linked_df[field].isna().sum()) if field in linked_df else 'missing column' for field in required_fields}
missing

## Results for Report

Use `entity_linking_summary.json` for concise report metrics such as number of linked subjects, linked objects, mapped properties, low-confidence records, and average linking confidence.

In [ ]:
json.loads(SUMMARY_PATH.read_text())

## Limitations

Entity linking remains uncertain. Some extracted triples contain vague subjects or objects, such as pronouns, generic noun phrases, or incomplete names. Those records are kept with low confidence instead of being removed, because downstream reasoning should know when evidence coverage is weak. Generic relations such as `say`, `be`, and `have` are not forced into Wikidata properties unless the claim pattern is specific enough.